# Descriptive analysis — predictions for estimation and transitions

This notebook loads the artefacts produced by `data_pipeline_v7.ipynb` and runs descriptives
designed to **predict, before estimation, what each regime's parameter estimates and the
transition paths between regimes are likely to look like**.

It mirrors the conventions of `data_pipeline_v7.ipynb` (Julia, `DataFrames`, `Arrow`, `Plots`)
and reads the same `data/derived/` directory.

**Single source of truth for windows.** This notebook does NOT define the four estimation
windows. Instead it reads `derived/windows.json`, which is written by `data_pipeline_v7.ipynb`.
Edit the windows once, in Stage 0 of the pipeline notebook; the change automatically
propagates here.

**Inputs read from `DERIVED_DIR`**

| File | Contents |
|---|---|
| `windows.json`                 | Window definitions (4 entries: ym_start, ym_end, asec_years, label) |
| `cps_basic_clean.arrow`        | Cleaned CPS Basic monthly micro-data with `window`, `skilled`, `employed`, `unemployed`, `in_training`, `in_lf`, `in_age`, `WTFINL` |
| `cps_asec_clean.arrow`         | Cleaned ASEC wage micro-data: `wage_norm`, `real_wage`, `skilled`, `window`, `ASECWT` |
| `transitions_monthly.arrow`    | Monthly job-finding / EU separation / LF→NILF / train-entry rates by skill |
| `transitions.arrow`            | Window-averaged transition rates |
| `jolts_clean.arrow`            | JOLTS vacancies allocated to skilled/unskilled by month |
| `j2j_ee_rates.csv`             | EE rate from J2J (E4-only) by window |
| `nu_estimation.csv`            | Demographic turnover rate $\nu$ on `base_fc` AND `base_covid` |
| `phi_calibration.csv`          | Training completion rate $\phi$ |
| `moments_{window}.csv`         | The 24 SMM moments per window |
| `sigma_{window}.csv`           | 24×24 variance–covariance matrix per window |
| `industry_skill_shares.arrow`  | Industry × window skill shares |
| `economy_skill_shares.arrow`   | Economy-wide skill share by window |

The four windows are read from `windows.json`. With the default v7 spec these are
`:base_fc`, `:crisis_fc`, `:base_covid`, `:crisis_covid`.


## 0. Setup

In [ ]:
using DataFrames
using CSV
using Arrow
using Statistics
using StatsBase
using LinearAlgebra
using Printf
using Plots
using StatsPlots
using Dates
using JSON3

gr()  # backend
default(fontfamily = "sans-serif", framestyle = :box, grid = true,
        gridalpha = 0.25, gridlinestyle = :dot, legendfontsize = 7,
        titlefontsize = 10, guidefontsize = 9, tickfontsize = 8)

# ── Paths ────────────────────────────────────────────────────────────────
# Adjust PROJECT_ROOT if this notebook lives somewhere other than code/notebooks/.
const PROJECT_ROOT = abspath(joinpath(@__DIR__, "..", ".."))
const DERIVED_DIR  = joinpath(PROJECT_ROOT, "data", "derived")

println("PROJECT_ROOT = ", PROJECT_ROOT)
println("DERIVED_DIR  = ", DERIVED_DIR, "  (exists: ", isdir(DERIVED_DIR), ")")

# ── Load WINDOWS from data_pipeline_v7's windows.json ────────────────────
# This is the SINGLE place windows are read into this notebook. To change
# the four estimation periods, edit the WINDOWS dict in
# `data_pipeline_v7.ipynb` (Stage 0) and rerun it; the JSON will be
# regenerated and this notebook will pick up the change on next run.
const _WIN_JSON_PATH = joinpath(DERIVED_DIR, "windows.json")
@assert isfile(_WIN_JSON_PATH) "Missing windows.json — run data_pipeline_v7 first"

const _win_raw = JSON3.read(read(_WIN_JSON_PATH, String))

const WINDOWS = Dict{Symbol, NamedTuple}(
    Symbol(k) => (
        label      = String(v["label"]),
        ym_start   = Int(v["ym_start"]),
        ym_end     = Int(v["ym_end"]),
        asec_years = Int(first(v["asec_years"])):Int(last(v["asec_years"])),
    )
    for (k, v) in _win_raw["windows"]
)

const WINDOWS_ORDER = Symbol.(_win_raw["windows_order"])

const WINDOW_LABEL = Dict(w => WINDOWS[w].label for w in keys(WINDOWS))

const WINDOW_COLOR = Dict(
    :base_fc      => RGB(0.306, 0.475, 0.655),
    :crisis_fc    => RGB(0.882, 0.341, 0.349),
    :base_covid   => RGB(0.349, 0.631, 0.310),
    :crisis_covid => RGB(0.949, 0.557, 0.169),
)

# Shared assign_window helper, sourced from the loaded WINDOWS dict so
# every cell uses the same boundaries. Replaces the per-cell hard-coded
# assign_window helpers that lived in v1.
function assign_window_from_ym(ym::Integer)::Symbol
    for w in WINDOWS_ORDER
        wd = WINDOWS[w]
        wd.ym_start <= ym <= wd.ym_end && return w
    end
    return :none
end

println("Loaded WINDOWS:")
for w in WINDOWS_ORDER
    wd = WINDOWS[w]
    println("  ", w, "  ", wd.label, "  ", wd.ym_start, "..", wd.ym_end)
end


In [ ]:
# ── Loaders ──────────────────────────────────────────────────────────────

function read_arrow_safe(name::AbstractString)
    p = joinpath(DERIVED_DIR, name)
    if !isfile(p)
        @warn "missing file" file=name
        return nothing
    end
    return DataFrame(Arrow.Table(p))
end

function read_csv_safe(name::AbstractString)
    p = joinpath(DERIVED_DIR, name)
    if !isfile(p)
        @warn "missing file" file=name
        return nothing
    end
    return CSV.read(p, DataFrame)
end

cps_basic   = read_arrow_safe("cps_basic_clean.arrow")
cps_asec    = read_arrow_safe("cps_asec_clean.arrow")
trans_m     = read_arrow_safe("transitions_monthly.arrow")
trans_w     = read_arrow_safe("transitions.arrow")
jolts       = read_arrow_safe("jolts_clean.arrow")
ind_shares  = read_arrow_safe("industry_skill_shares.arrow")
econ_shares = read_arrow_safe("economy_skill_shares.arrow")

j2j_ee   = read_csv_safe("j2j_ee_rates.csv")
nu_est   = read_csv_safe("nu_estimation.csv")
phi_cal  = read_csv_safe("phi_calibration.csv")

# Coerce :window columns to Symbol where present
for df in (cps_basic, cps_asec, trans_m, trans_w, jolts, ind_shares, econ_shares, j2j_ee)
    if df !== nothing && hasproperty(df, :window) && !(eltype(df.window) <: Symbol)
        df.window = Symbol.(df.window)
    end
end

moments = Dict{Symbol, DataFrame}()
for w in WINDOWS_ORDER
    df = read_csv_safe("moments_$(w).csv")
    df === nothing && continue
    moments[w] = df
end

sigmas = Dict{Symbol, Matrix{Float64}}()
sigma_names = Dict{Symbol, Vector{String}}()
for w in WINDOWS_ORDER
    p = joinpath(DERIVED_DIR, "sigma_$(w).csv")
    isfile(p) || continue
    raw = CSV.read(p, DataFrame)
    # Pipeline saves DataFrame(Sigma, MOMENT_NAMES): all K columns numeric,
    # moment names live in the header — there is NO row-name column.
    sigmas[w]      = Matrix{Float64}(raw)
    sigma_names[w] = string.(names(raw))
end

println("Moments loaded: ", collect(keys(moments)))
println("Sigmas loaded:  ", collect(keys(sigmas)))
println("ν estimates:")
nu_est === nothing || display(nu_est)


In [ ]:
# ── Weighted statistics helpers (mirrors the data_processing/*.jl pipeline stages) ─────────

function wmean(x::AbstractVector, w::AbstractVector)
    sw = sum(w); sw <= 0.0 && return NaN
    return sum(x .* w) / sw
end

function wvar(x::AbstractVector, w::AbstractVector)
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^2) / sw
end

wsd(x, w) = sqrt(max(wvar(x, w), 0.0))

function wcm3(x::AbstractVector, w::AbstractVector)
    m = wmean(x, w); sw = sum(w)
    sw <= 0.0 && return NaN
    return sum(w .* (x .- m).^3) / sw
end

function wquantile(x::AbstractVector, w::AbstractVector, q::Real)
    n = length(x); n == 0 && return NaN
    ord = sortperm(x)
    xs, ws = x[ord], w[ord]
    cs = cumsum(ws); total = cs[end]
    total <= 0.0 && return NaN
    target = q * total
    i = searchsortedfirst(cs, target)
    i = clamp(i, 1, length(xs))
    return Float64(xs[i])
end

# Strip non-finite rows (mirrors the pipeline's `filter(isfinite, ...)`)
finite_mean(v) = isempty(v) ? NaN : mean(filter(isfinite, v))

println("Helpers ready.")

## 1. Sample diagnostics

Sanity-check the sample sizes per window. The variance–covariance matrix $\widehat\Sigma$ is
built from these counts, so big imbalances translate directly into the SMM weighting matrix
and into how informative each window is for the deep block.

**What to look for**

- CPS Basic LF observations should be in the millions per window, with crisis windows naturally
  shorter (and so smaller).
- ASEC counts should be tens of thousands of wage observations per window.
- A drop in skilled-share between baseline and crisis is *not* expected; the skill margin moves
  slowly. A big drop is a red flag for a definition change.

In [ ]:
function sample_table(df, weight_col::Union{Symbol, Nothing}, label::AbstractString)
    df === nothing && return DataFrame(window=Symbol[], n_rows=Int[], weighted_n=Float64[], source=String[])
    rows = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), df)
        n   = nrow(sub)
        wn  = weight_col === nothing || !hasproperty(sub, weight_col) ? NaN :
              sum(coalesce.(sub[!, weight_col], 0.0))
        push!(rows, (window=w, n_rows=n, weighted_n=wn, source=label))
    end
    return DataFrame(rows)
end

tbl = vcat(
    sample_table(cps_basic, :WTFINL,   "CPS Basic"),
    sample_table(cps_asec,  :ASECWT,   "CPS ASEC"),
    sample_table(trans_m,   :n_pairs,  "Transitions (monthly)"),
    sample_table(jolts,     nothing,   "JOLTS"),
)

# Pivot rows = window, cols = source (n_rows)
piv = unstack(tbl, :window, :source, :n_rows)
piv = piv[indexin(WINDOWS_ORDER, piv.window), :]
display(piv)

## 2. Cross-window moment table

The most important descriptive in the notebook. Every SMM moment is shown side-by-side across
the four windows, with the **percent change baseline → crisis** for both the FC and the COVID
episodes.

**Patterns to look for, mapped to the model**

| Pattern in the data | Tells you about |
|---|---|
| Big jump in `ur_U`, `ur_S` between baseline and crisis | Job-finding / separation block ($\lambda_j(z)$, $\xi_S(z)$, $k_j(z)$) |
| `ur_U` rises **more** than `ur_S` in FC, but `ur_S` and `ur_U` rise similarly in COVID | Asymmetric vs. symmetric productivity shock — informs the relative move of $P_U(z)$ vs. $P_S(z)$ |
| `mean_wage_U`/`mean_wage_S` ratio stable while `wage_premium` moves | Composition vs. price effect; bargaining vs. productivity |
| `theta_U`, `theta_S` collapse | Vacancy-cost $k_j(z)$ rising or surplus collapsing |
| `training_share` rises in crisis | Training cutoff $\bar x(z)$ falling — return to skilled rises faster than return to unskilled |
| `emp_var_U`, `emp_cm3_U` widen | Damage shape $\alpha_U(z)$ falling (heavier left tail) |
| `ee_rate_S` collapses | Skilled job ladder freezing (lower $\theta_S$, lower $\Gamma_z$ tail) |

In [ ]:
if !isempty(moments)
    M = DataFrame(moment = moments[first(keys(moments))].moment)
    for w in WINDOWS_ORDER
        haskey(moments, w) || continue
        M[!, String(w)] = moments[w].value
    end
    if "base_fc" in names(M) && "crisis_fc" in names(M)
        M[!, "Δ% FC"]    = (M.crisis_fc    ./ M.base_fc    .- 1.0) .* 100
    end
    if "base_covid" in names(M) && "crisis_covid" in names(M)
        M[!, "Δ% COVID"] = (M.crisis_covid ./ M.base_covid .- 1.0) .* 100
    end
    display(M)
end

In [ ]:
# Visualise the deltas — which moments move the most between regimes?
if !isempty(moments) && haskey(moments, :base_fc) && haskey(moments, :crisis_fc) &&
   haskey(moments, :base_covid) && haskey(moments, :crisis_covid)

    names_ = moments[:base_fc].moment
    d_fc    = (moments[:crisis_fc].value    ./ moments[:base_fc].value    .- 1) .* 100
    d_covid = (moments[:crisis_covid].value ./ moments[:base_covid].value .- 1) .* 100

    # One ordering: sort by |FC delta| ascending (largest at the top).
    # Apply the same ordering to COVID.
    ord = sortperm(abs.(d_fc))
    n   = length(ord)
    lab = String.(names_[ord])
    y   = collect(1:n)

    fc_sorted    = d_fc[ord]
    covid_sorted = d_covid[ord]

    # Shared x-limits across both subplots
    all_vals = filter(isfinite, vcat(fc_sorted, covid_sorted))
    pad      = 0.05 * (maximum(all_vals) - minimum(all_vals))
    xlims_   = (minimum(all_vals) - pad, maximum(all_vals) + pad)
    ylims_   = (0.5, n + 0.5)

    p1 = bar(y, fc_sorted;
             orientation = :h,
             yticks      = (y, lab),
             ylims       = ylims_,
             xlims       = xlims_,
             color       = WINDOW_COLOR[:crisis_fc],
             legend      = false,
             title       = "% change baseline → FC",
             xlabel      = "% change")
    vline!(p1, [0]; color = :black, lw = 0.6, label = "")

    p2 = bar(y, covid_sorted;
             orientation = :h,
             yticks      = (y, lab),
             ylims       = ylims_,
             xlims       = xlims_,
             color       = WINDOW_COLOR[:crisis_covid],
             legend      = false,
             title       = "% change baseline → COVID",
             xlabel      = "% change")
    vline!(p2, [0]; color = :black, lw = 0.6, label = "")

    display(plot(p1, p2;
                 layout       = (1, 2),
                 link         = :y,
                 size         = (1200, 750),
                 left_margin  = 10Plots.mm,
                 bottom_margin = 4Plots.mm))
end

## 3. Time series spanning the regime changes

The model's transition section says that, after a regime switch, value functions adjust
*immediately* but cross-sectional densities adjust *gradually* — with the slow clock
$\nu + \phi$ governing the skilled-share half-life.

The right place to look in the data is monthly time series that cross the regime boundary:

1. unemployment rates by skill;
2. skilled share of the labour force;
3. training share;
4. market tightness $\theta_U$, $\theta_S$;
5. monthly job-finding and separation rates.

**Predictions**

- $UR$ should jump within 1–2 quarters and then decay over months — fastest of the three clocks.
- Skilled share should barely move on impact and then drift over years.
- Training share should jump on impact in regimes where the relative return to skilled rises
  (FC: predicted rise; COVID: ambiguous).
- Tightness should collapse with $UR$, but its recovery shape is informative about $k_j(z)$
  and the surplus-distribution channel.

In [ ]:
function monthly_ts_from_cps(df)
    df === nothing && return nothing

    # Two passes per (YEAR, MONTH):
    #   • LF ∩ ¬train denominator for ur_total, ur_U, ur_S, skilled_share
    #     (matches v7 stock-moment denominators).
    #   • Working-age population (the full age 16-64 cps_basic sample) for
    #     the strict training_share = (NILF ∩ train) / pop.
    has_train = hasproperty(df, :in_training)

    df_all = transform(df, [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    lf_excl_train = if has_train && hasproperty(df_all, :in_lf)
        filter(r -> r.in_lf && !r.in_training, df_all)
    elseif hasproperty(df_all, :in_lf)
        filter(:in_lf => identity, df_all)
    else
        df_all
    end

    rows = NamedTuple[]
    # First pass: LF ∩ ¬train denominator
    for sub_g in groupby(lf_excl_train, [:ym, :window])
        sub = DataFrame(sub_g)
        w   = Float64.(coalesce.(sub.WTFINL, 0.0))
        sw  = sum(w)
        sw <= 0 && continue
        u    = sum(w[sub.unemployed])
        u_S  = sum(w[sub.unemployed .&  sub.skilled])
        u_U  = sum(w[sub.unemployed .& .!sub.skilled])
        lf_S = sum(w[ sub.skilled])
        lf_U = sum(w[.!sub.skilled])
        push!(rows, (
            ym             = sub.ym[1],
            window         = sub.window[1],
            ur_total       = u  / sw,
            ur_U           = lf_U > 0 ? u_U / lf_U : NaN,
            ur_S           = lf_S > 0 ? u_S / lf_S : NaN,
            skilled_share  = lf_S / sw,
        ))
    end
    out = DataFrame(rows)

    # Second pass: training_share strict variant (NILF ∩ train / pop)
    if has_train && hasproperty(df_all, :in_lf)
        ts_rows = NamedTuple[]
        for sub_g in groupby(df_all, [:ym, :window])
            sub = DataFrame(sub_g)
            w   = Float64.(coalesce.(sub.WTFINL, 0.0))
            pop = sum(w)
            pop <= 0 && continue
            num = sum(w[coalesce.(sub.in_training, false) .& .!sub.in_lf])
            push!(ts_rows, (ym = sub.ym[1], window = sub.window[1],
                            training_share = num / pop))
        end
        ts_df = DataFrame(ts_rows)
        out = leftjoin(out, ts_df, on = [:ym, :window])
    else
        out.training_share = fill(NaN, nrow(out))
    end

    sort!(out, :ym)
    out.date = Date.(div.(out.ym, 100), mod.(out.ym, 100), 1)
    return out
end

ts = monthly_ts_from_cps(cps_basic)
ts === nothing || display(first(ts, 5))


In [ ]:
# ── Era-based regime-switch helpers (shared by the time-series plots below) ──
const ERA_SWITCH  = Dict(:fc => 200801, :covid => 202001)
const ERA_WINDOWS = Dict(
    :fc    => (base = :base_fc,    crisis = :crisis_fc),
    :covid => (base = :base_covid, crisis = :crisis_covid),
)
# FC drawn in blues, COVID drawn in reds; base is the lighter shade of each
const ERA_COLOR = Dict(
    :fc    => (base = RGB(0.55, 0.72, 0.92), crisis = RGB(0.10, 0.20, 0.55)),
    :covid => (base = RGB(0.96, 0.62, 0.55), crisis = RGB(0.62, 0.08, 0.12)),
)
const ERA_LABEL = Dict(:fc => "FC", :covid => "COVID")

function months_from_switch(ym::Integer, era::Symbol)
    sw = ERA_SWITCH[era]
    yd, md = divrem(ym, 100)
    ys, ms = divrem(sw, 100)
    return (yd - ys) * 12 + (md - ms)
end

# Plot each series with FC and COVID overlaid on a "months from regime switch"
# axis. Within each era the base segment is light, the crisis segment is dark,
# and the two are joined (no break at the regime change). A black dashed line
# marks the event. 
function plot_ts(ts::DataFrame, cols::Vector{Symbol}, titles::Vector{String};
                 legend_pos = :topleft)
    plt = plot(layout = (length(cols), 1),
               size = (850, 290 * length(cols)),
               left_margin   = 6Plots.mm,
               right_margin  = 4Plots.mm,
               bottom_margin = 5Plots.mm,
               top_margin    = 3Plots.mm)
    for (i, (col, ttl)) in enumerate(zip(cols, titles))
        for era in (:fc, :covid)
            base_w = ERA_WINDOWS[era].base
            cris_w = ERA_WINDOWS[era].crisis
            sub = sort(filter(:window => w -> w == base_w || w == cris_w, ts), :ym)
            isempty(sub) && continue
            tvec    = months_from_switch.(sub.ym, era)
            base_ix = findall(sub.window .== base_w)
            cris_ix = findall(sub.window .== cris_w)
            # Bridge: include the first crisis point at the end of the base
            # segment so the two line pieces meet (no visible break).
            base_plot = isempty(cris_ix) ? base_ix : vcat(base_ix, first(cris_ix))
            if !isempty(base_plot)
                plot!(plt, tvec[base_plot], sub[base_plot, col];
                      subplot = i, color = ERA_COLOR[era].base, lw = 1.5,
                      label = "", title = ttl)
            end
            if !isempty(cris_ix)
                # Label only on the first subplot so the whole figure has
                # one consolidated legend (just FC / COVID, crisis colors).
                lbl = i == 1 ? ERA_LABEL[era] : ""
                plot!(plt, tvec[cris_ix], sub[cris_ix, col];
                      subplot = i, color = ERA_COLOR[era].crisis, lw = 1.8,
                      label = lbl)
            end
        end
        vline!(plt, [0]; subplot = i, color = :black, lw = 0.8, ls = :dash,
               label = "")
        plot!(plt; subplot = i,
              xlabel = "months from regime switch",
              legend = i == 1 ? legend_pos : false)
    end
    display(plt)
end

if ts !== nothing
    plot_ts(ts,
            [:ur_total, :ur_U, :ur_S],
            ["Unemployment rate — all", "Unskilled UR", "Skilled UR"];
            legend_pos = :topleft)            # inside, top-right of subplot 1
    plot_ts(ts,
            [:skilled_share, :training_share],
            ["Skilled share of LF (slow clock)", "Training share of LF"];
            legend_pos = :topleft)          # outside, parked to the right
end

In [ ]:
# Tightness time series (V/U) by skill — overlaid on the months-from-switch axis
if jolts !== nothing && cps_basic !== nothing
    cps_um = transform(filter(:in_lf => identity, cps_basic),
                       [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    ucounts = combine(groupby(cps_um, :ym),
        [:WTFINL, :unemployed, :skilled] => ((w, u, s) -> begin
            ww = Float64.(coalesce.(w, 0.0))
            (U_U = sum(ww[u .& .!s]), U_S = sum(ww[u .& s]))
        end) => AsTable)
    j  = transform(jolts, [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    jt = innerjoin(j, ucounts, on = :ym)
    jt.theta_U = ifelse.(jt.U_U .> 0, jt.V_U ./ jt.U_U, NaN)
    jt.theta_S = ifelse.(jt.U_S .> 0, jt.V_S ./ jt.U_S, NaN)
    jt.date    = Date.(div.(jt.ym, 100), mod.(jt.ym, 100), 1)

    # Window assignment from the loaded WINDOWS dict (no hard-coded bounds)
    if !hasproperty(jt, :window)
        jt.window = Symbol.(assign_window_from_ym.(jt.ym))
    end

    plot_ts(jt, [:theta_U, :theta_S],
            ["θ_U (V/U)", "θ_S (V/U)"])
end


In [ ]:
# Job-finding & separation rates by skill (monthly transitions)
# Real calendar dates on the x-axis. Within each era (FC, COVID) the line is
# drawn with the base colour during the base window and the crisis colour
# during the crisis window, with the two segments joined at the regime switch.
if trans_m !== nothing
    tm = transform(trans_m, [:year, :month] => ((y, m) -> 100 .* y .+ m) => :ym)
    tm.date = Date.(div.(tm.ym, 100), mod.(tm.ym, 100), 1)

    plt = plot(layout = (2, 2), size = (1200, 720),
               left_margin   = 6Plots.mm,
               right_margin  = 4Plots.mm,
               bottom_margin = 5Plots.mm,
               top_margin    = 3Plots.mm)
    for (j, sk) in enumerate((false, true))
        for (i, col) in enumerate((:jfr, :sep))
            spi = (i - 1) * 2 + j
            for era in (:fc, :covid)
                base_w = ERA_WINDOWS[era].base
                cris_w = ERA_WINDOWS[era].crisis
                sub = sort(filter([:skilled, :window] =>
                                  (s, w) -> s == sk && (w == base_w || w == cris_w),
                                  tm), :ym)
                isempty(sub) && continue
                base_ix = findall(sub.window .== base_w)
                cris_ix = findall(sub.window .== cris_w)
                base_plot = isempty(cris_ix) ? base_ix : vcat(base_ix, first(cris_ix))
                if !isempty(base_plot)
                    plot!(plt, sub.date[base_plot], sub[base_plot, col];
                          subplot = spi, color = ERA_COLOR[era].base, lw = 1.3,
                          label = "")
                end
                if !isempty(cris_ix)
                    # Label only on subplot 1, crisis colors only → 2-entry legend.
                    lbl = spi == 1 ? ERA_LABEL[era] : ""
                    plot!(plt, sub.date[cris_ix], sub[cris_ix, col];
                          subplot = spi, color = ERA_COLOR[era].crisis, lw = 1.6,
                          label = lbl)
                end
            end
            plot!(plt; subplot = spi,
                  title = string(sk ? "Skilled" : "Unskilled", " — ",
                                 col == :jfr ? "JFR" : "EU sep rate"),
                  legend = (spi == 1 ? :bottomleft : false))
        end
    end
    display(plt)
end

## 4. Transition-dynamics descriptives

The model says, after a regime switch at $T$:

- **Fast clock** (months): job-finding $f_j$, separation $\delta_j$ → unemployment rates revert.
- **Medium clock** ($1/(\phi+\nu) \approx 14$ months for $\phi = 1/48$, $\nu \approx 0.005$):
  the training pipeline drains.
- **Slow clock** ($1/\nu$, decades): the skilled stock $M_S$.

We estimate, from the data, the empirical adjustment speed of each of these objects during FC
and COVID. If the data reject the model's separation of clocks, that's a structural
misspecification we want to know about *before* doing SMM.

In [ ]:
# Fit y_t - y_ss = (y_0 - y_ss) * exp(-r t)  →  r and half-life
function fit_decay(s::AbstractVector, baseline_value::Real)
    s = Float64.(s)
    s = s[isfinite.(s)]
    length(s) < 4 && return (NaN, NaN)
    dev = abs.(s .- baseline_value)
    dev = max.(dev, 1e-9)
    t   = collect(0.0:length(dev)-1)
    X = hcat(ones(length(t)), t)
    β = X \ log.(dev)
    r = -β[2]
    hl = r > 0 ? log(2) / r : NaN
    return (r, hl)
end

if ts !== nothing
    rows = NamedTuple[]
    for (base, crisis) in ((:base_fc, :crisis_fc), (:base_covid, :crisis_covid))
        for col in (:ur_total, :ur_U, :ur_S, :skilled_share, :training_share)
            base_val = mean(filter(isfinite, ts[ts.window .== base, col]))
            sub = sort(ts[ts.window .== crisis, :], :ym)
            r, hl = fit_decay(sub[!, col], base_val)
            push!(rows, (
                crisis = crisis, moment = col,
                baseline_mean = base_val,
                crisis_mean   = mean(filter(isfinite, sub[!, col])),
                decay_rate_r  = r,
                half_life_mo  = hl,
            ))
        end
    end
    decay_tbl = DataFrame(rows)
    display(decay_tbl)
    println()
    println("Reading: low decay_rate / long half_life ⇒ slow-clock object (skilled_share, training_share).")
    println("        Fast revert ⇒ ur_* — value-function-dominated, distribution moves with it.")
end

In [ ]:
# Composite plot:
#   (a) UR total (ur_total), FC vs COVID
#   (b) Market tightness: theta_U and theta_S, FC vs COVID — all four lines
# Bottom row: skilled share and training share of LF.
# Colours use ERA_COLOR (same as plot_ts): FC blues, COVID reds; base segment
# light, crisis segment dark. theta_U is solid, theta_S is dashed.
if ts !== nothing && jolts !== nothing && cps_basic !== nothing

    # Rebuild the tightness series (theta_U, theta_S) with window labels
    cps_um = transform(filter(:in_lf => identity, cps_basic),
                       [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    ucounts = combine(groupby(cps_um, :ym),
        [:WTFINL, :unemployed, :skilled] => ((w, u, s) -> begin
            ww = Float64.(coalesce.(w, 0.0))
            (U_U = sum(ww[u .& .!s]), U_S = sum(ww[u .& s]))
        end) => AsTable)
    j  = transform(jolts, [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    jt = innerjoin(j, ucounts, on = :ym)
    jt.theta_U = ifelse.(jt.U_U .> 0, jt.V_U ./ jt.U_U, NaN)
    jt.theta_S = ifelse.(jt.U_S .> 0, jt.V_S ./ jt.U_S, NaN)
    if !hasproperty(jt, :window)
        jt.window = Symbol.(assign_window_from_ym.(jt.ym))
    end

    lay = @layout [a b; c d]
    plt = plot(layout = lay, size = (1350, 780),
               left_margin   = 7Plots.mm,
               right_margin  = 4Plots.mm,
               top_margin    = 5Plots.mm,
               bottom_margin = 9Plots.mm)

    function draw_era!(df, col, era, i; ls = :solid, lbl = "")
        base_w = ERA_WINDOWS[era].base
        cris_w = ERA_WINDOWS[era].crisis
        sub = sort(filter(:window => w -> w == base_w || w == cris_w, df), :ym)
        isempty(sub) && return
        tvec    = months_from_switch.(sub.ym, era)
        base_ix = findall(sub.window .== base_w)
        cris_ix = findall(sub.window .== cris_w)
        base_plot = isempty(cris_ix) ? base_ix : vcat(base_ix, first(cris_ix))
        if !isempty(base_plot)
            plot!(plt, tvec[base_plot], sub[base_plot, col];
                  subplot = i, color = ERA_COLOR[era].base, lw = 1.6,
                  ls = ls, label = "")
        end
        if !isempty(cris_ix)
            plot!(plt, tvec[cris_ix], sub[cris_ix, col];
                  subplot = i, color = ERA_COLOR[era].crisis, lw = 1.8,
                  ls = ls, label = lbl)
        end
    end

    # (a) UR total
    draw_era!(ts, :ur_total, :fc,    1; ls = :solid, lbl = "FC")
    draw_era!(ts, :ur_total, :covid, 1; ls = :solid, lbl = "COVID")
    vline!(plt, [0]; subplot = 1, color = :black, lw = 0.6, ls = :dot, label = "")
    plot!(plt; subplot = 1, title = "UR — total",
          xlabel = "months from switch", legend = :topleft)

    # (b) Market tightness
    draw_era!(jt, :theta_U, :fc,    2; ls = :solid)
    draw_era!(jt, :theta_U, :covid, 2; ls = :solid)
    draw_era!(jt, :theta_S, :fc,    2; ls = :dash)
    draw_era!(jt, :theta_S, :covid, 2; ls = :dash)
    vline!(plt, [0]; subplot = 2, color = :black, lw = 0.6, ls = :dot, label = "")
    plot!(plt, [NaN], [NaN]; subplot = 2, color = :gray40, lw = 1.6,
          ls = :solid, label = "θ_U (unskilled)")
    plot!(plt, [NaN], [NaN]; subplot = 2, color = :gray40, lw = 1.6,
          ls = :dash,  label = "θ_S (skilled)")
    plot!(plt; subplot = 2, title = "Market tightness θ (V/U)",
          xlabel = "months from switch", legend = :topleft)

    # (c, d) skilled share, training share
    for (i, (col, ttl)) in enumerate(zip((:skilled_share, :training_share),
                                         ("Skilled share of LF", "Training share (NILF ∩ train / pop)")))
        spi = i + 2
        draw_era!(ts, col, :fc,    spi; ls = :solid)
        draw_era!(ts, col, :covid, spi; ls = :solid)
        vline!(plt, [0]; subplot = spi, color = :black, lw = 0.6, ls = :dot, label = "")
        plot!(plt; subplot = spi, title = ttl,
              xlabel = "months from switch", legend = false)
    end

    display(plt)
end


## 5. Beveridge curve by skill

The Beveridge curve is informative about whether vacancy creation drops more than separations
rise. In the model:

- A productivity shock alone (lower $P_j$) shifts the economy *along* the curve.
- A change in matching efficiency $\mu_j$, or a structural mismatch shock, shifts the curve.

Because the deep block treats $\mu_j$ as fixed, the Beveridge curves of the two crisis windows
should *trace out a movement along* the baseline curve, not a parallel outward shift. A clearly
shifted curve is evidence that holding $\mu_j$ deep is too restrictive.

In [ ]:
if jolts !== nothing && cps_basic !== nothing
    # ── data prep ─────────────────────────────────────────────────────
    cps_um = transform(filter(:in_lf => identity, cps_basic),
                       [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)

    ucounts = combine(groupby(cps_um, :ym),
        [:WTFINL, :unemployed, :skilled] => ((w, u, s) -> begin
            ww = Float64.(coalesce.(w, 0.0))
            sw = sum(ww)
            u_U = sum(ww[u .& .!s])
            u_S = sum(ww[u .& s])
            lf_U = sum(ww[.!s]); lf_S = sum(ww[s])
            (U_U = u_U, U_S = u_S,
             ur_U = lf_U > 0 ? u_U / lf_U : NaN,
             ur_S = lf_S > 0 ? u_S / lf_S : NaN)
        end) => AsTable)

    j   = transform(jolts, [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    bev = innerjoin(j, ucounts, on = :ym)

    if !hasproperty(bev, :window)
        bev.window = Symbol.(assign_window_from_ym.(bev.ym))
    end

    series     = ((:V_U, :ur_U, "U"), (:V_S, :ur_S, "S"))
    bev_sorted = sort(bev, :ym)

    in_windows = in.(bev_sorted.window, Ref(Set(WINDOWS_ORDER)))
    bs_named   = bev_sorted[in_windows, :]

    function time_path_xy(df::DataFrame, ur_col::Symbol, V_col::Symbol)
        xs = Float64.(df[!, ur_col])
        ys = Float64.(df[!, V_col]) ./ 1e6
        ym_v = collect(df.ym)
        for k in length(ym_v):-1:2
            prev_y, prev_m = divrem(ym_v[k-1], 100)
            curr_y, curr_m = divrem(ym_v[k],   100)
            gap = (curr_y - prev_y) * 12 + (curr_m - prev_m)
            if gap > 1
                insert!(xs, k, NaN)
                insert!(ys, k, NaN)
            end
        end
        return xs, ys
    end

    # ── Plot 1: Beveridge scatters only (1 × 2: U, S) ─────────────────
    plt1 = plot(layout = (1, 2), size = (1200, 460),
                left_margin   = 8Plots.mm,
                right_margin  = 4Plots.mm,
                top_margin    = 3Plots.mm,
                bottom_margin = 7Plots.mm)
    for (i, (V_col, ur_col, lab)) in enumerate(series)
        for w in WINDOWS_ORDER
            sub = filter(:window => ==(w), bev)
            isempty(sub) && continue
            scatter!(plt1, sub[!, ur_col], sub[!, V_col] ./ 1e6;
                     subplot = i, ms = 3, mc = WINDOW_COLOR[w],
                     markerstrokewidth = 0, alpha = 0.75,
                     label  = String(w),
                     title  = "Beveridge curve — $lab",
                     xlabel = "unemployment rate — $lab",
                     ylabel = "vacancies V_$lab  (millions)")
        end
    end
    display(plt1)

    # ── Plot 2: Beveridge scatters + time path (1 × 2: U, S) ──────────
    plt2 = plot(layout = (1, 2), size = (1200, 460),
                left_margin   = 8Plots.mm,
                right_margin  = 4Plots.mm,
                top_margin    = 3Plots.mm,
                bottom_margin = 7Plots.mm)
    for (i, (V_col, ur_col, lab)) in enumerate(series)
        xs, ys = time_path_xy(bs_named, ur_col, V_col)
        plot!(plt2, xs, ys;
              subplot = i, color = RGB(0.45, 0.45, 0.45), lw = 0.6,
              alpha = 0.7, label = "time path")
        for w in WINDOWS_ORDER
            sub = sort(filter(:window => ==(w), bev), :ym)
            isempty(sub) && continue

            scatter!(plt2, sub[!, ur_col], sub[!, V_col] ./ 1e6;
                     subplot = i, ms = 3, mc = WINDOW_COLOR[w],
                     markerstrokewidth = 0, alpha = 0.85,
                     label  = String(w),
                     title  = "Beveridge curve with time path — $lab",
                     xlabel = "unemployment rate — $lab",
                     ylabel = "vacancies V_$lab  (millions)")

            if w in (:crisis_fc, :crisis_covid)
                ok = isfinite.(sub[!, ur_col]) .& isfinite.(sub[!, V_col])
                ws = sub[ok, :]
                isempty(ws) && continue
                scatter!(plt2, [ws[1, ur_col]], [ws[1, V_col] / 1e6];
                         subplot = i, markershape = :circle,
                         ms = 9, mc = WINDOW_COLOR[w],
                         markerstrokewidth = 1.0, markerstrokecolor = :white,
                         label = "")
                scatter!(plt2, [ws[end, ur_col]], [ws[end, V_col] / 1e6];
                         subplot = i, markershape = :star5,
                         ms = 11, mc = WINDOW_COLOR[w],
                         markerstrokewidth = 1.0, markerstrokecolor = :white,
                         label = "")
            end
        end
        scatter!(plt2, [NaN], [NaN]; subplot = i, markershape = :circle,
                 ms = 9, mc = :gray40, markerstrokewidth = 1.0,
                 markerstrokecolor = :white, label = "start")
        scatter!(plt2, [NaN], [NaN]; subplot = i, markershape = :star5,
                 ms = 11, mc = :gray40, markerstrokewidth = 1.0,
                 markerstrokecolor = :white, label = "end")
    end
    display(plt2)
end


## 6. SMM weighting matrix diagnostics

The SMM objective is
$$
\widehat\Theta = \arg\min_{\Theta}\, (m_{\rm data} - m_{\rm model}(\Theta))^\top\, W\, (m_{\rm data} - m_{\rm model}(\Theta)),
$$
with $W = \widehat\Sigma^{-1}$ in the optimal weighting. Diagnostics on $\widehat\Sigma$ tell us
which moments will *drive* the estimation:

- moments with very small standard errors (small diagonal of $\Sigma$) get high weight;
- moment pairs with high correlation in $\Sigma$ are partially redundant.

In [ ]:
sigmas = Dict{Symbol, Matrix{Float64}}()
sigma_names = Dict{Symbol, Vector{String}}()
for w in WINDOWS_ORDER
    p = joinpath(DERIVED_DIR, "sigma_$(w).csv")
    isfile(p) || continue
    raw = CSV.read(p, DataFrame)
    # Pipeline saves DataFrame(Sigma, MOMENT_NAMES): all K columns numeric,
    # moment names live in the header — there is NO row-name column.
    sigmas[w]      = Matrix{Float64}(raw)
    sigma_names[w] = string.(names(raw))
end

In [ ]:
if haskey(sigmas, :base_fc)
    S = sigmas[:base_fc]
    nm = sigma_names[:base_fc]
    sd = sqrt.(diag(S))
    corr = S ./ (sd * sd')
    plt = heatmap(corr; clim = (-1, 1), c = :RdBu,
                  xticks = (1:length(nm), nm), yticks = (1:length(nm), nm),
                  xrotation = 90, size = (760, 700),
                  title = "Moment correlation matrix — base_fc",
                  tickfontsize = 6)
    display(plt)
end

## 7. Synthesis — predictions for the estimation and transition

| Block | Parameter | Descriptive cue | Prediction |
|---|---|---|---|
| Deep | $a_\ell, b_\ell$ | §5 — rank distribution of unskilled wages, base_fc | Method-of-moments fit on rank gives a starting estimate |
| Deep | $\eta_U, \eta_S$ | §6 — log $f$ vs log $\theta$ slope | $1 -$ slope; should be ≈ $0.5$ |
| Deep | $\mu_U, \mu_S$ | §6 — intercept of the same regression | $\exp(\text{intercept})$ |
| Deep | $\beta_U, \beta_S$ | mean wage / mean productivity ratio | Ballpark check on Hosios |
| Deep | $b_U, b_S$ (outside option) | p10 of `wage_norm`; UR levels | Outside option ≈ lower-tail consumption value |
| Deep | $c$ | training_share level in baseline | Higher training_share ⇒ lower $c$ |
| Regime | $\gamma^{P_U}, P_S$ | $\Delta\%$ mean_wage in §2 | Direct |
| Regime | $\lambda_U, \lambda_S$ | $\Delta$ separation rates in §2/§3 | Direct |
| Regime | $\xi_S$ | $\Delta$ skilled separation rate | Direct |
| Regime | $\alpha_U$ | $\Delta$ wage variance / skewness for unskilled | Wider tail ⇒ smaller $\alpha_U$ |
| Regime | $a_\Gamma, b_\Gamma$ | Quantile shifts in skilled wages (§4) | Direct |
| Regime | $k_U, k_S$ | $\Delta\theta$ vs $\Delta f$ — shifts on the matching curve | $k$ rises ⇒ θ falls more than $f$ |
| **Transition** | clock $\nu$ | half-life of skilled_share in §7 | Should be ≈ $\ln 2 / \nu$ months |
| **Transition** | clock $\phi+\nu$ | half-life of training_share in §7 | Should be ≈ $14$ months for $\phi = 1/48$ |
| **Transition** | fast clock | half-life of UR series in §7 | A few months — value-function clock |

In [ ]:
# Calibrated values used as transition-clock benchmarks
nu_est  === nothing || (println("ν estimates (life-table, one row per baseline window):"); display(nu_est))
phi_cal === nothing || (println("\nφ calibration (pooled across Fall semesters):"); display(phi_cal))


## Diagnostics

In [ ]:
# -----------------------------------------------------------------------------
# CELL D1.  Beveridge-curve table — quarterly, with dates and window labels
# -----------------------------------------------------------------------------
# Output: a DataFrame `bev_q` with one row per (year, quarter) showing
#   ym, date, quarter, window, V_U, V_S, U_U, U_S, ur_U, ur_S, theta_U, theta_S.

if jolts !== nothing && cps_basic !== nothing
    cps_um = transform(filter(:in_lf => identity, cps_basic),
                       [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    ucounts = combine(groupby(cps_um, :ym),
        [:WTFINL, :unemployed, :skilled] => ((w, u, s) -> begin
            ww = Float64.(coalesce.(w, 0.0))
            sw = sum(ww)
            u_U = sum(ww[u .& .!s])
            u_S = sum(ww[u .& s])
            lf_U = sum(ww[.!s]); lf_S = sum(ww[s])
            (U_U = u_U, U_S = u_S,
             ur_U = lf_U > 0 ? u_U / lf_U : NaN,
             ur_S = lf_S > 0 ? u_S / lf_S : NaN)
        end) => AsTable)

    j   = transform(jolts, [:YEAR, :MONTH] => ((y, m) -> 100 .* y .+ m) => :ym)
    bev = innerjoin(j, ucounts, on = :ym)

    bev.window  = Symbol.(assign_window_from_ym.(bev.ym))
    bev.year    = div.(bev.ym, 100)
    bev.month   = mod.(bev.ym, 100)
    bev.quarter = ceil.(Int, bev.month ./ 3)
    bev.theta_U = ifelse.(bev.U_U .> 0, bev.V_U ./ bev.U_U, NaN)
    bev.theta_S = ifelse.(bev.U_S .> 0, bev.V_S ./ bev.U_S, NaN)

    # Quarterly averages
    bev_q = combine(groupby(bev, [:year, :quarter, :window]),
        :V_U     => mean => :V_U,
        :V_S     => mean => :V_S,
        :U_U     => mean => :U_U,
        :U_S     => mean => :U_S,
        :ur_U    => mean => :ur_U,
        :ur_S    => mean => :ur_S,
        :theta_U => mean => :theta_U,
        :theta_S => mean => :theta_S,
    )
    sort!(bev_q, [:year, :quarter])
    bev_q.date_label = string.(bev_q.year, "Q", bev_q.quarter)

    bev_q.V_U_mn = bev_q.V_U ./ 1e6
    bev_q.V_S_mn = bev_q.V_S ./ 1e6
    bev_q.U_U_mn = bev_q.U_U ./ 1e6
    bev_q.U_S_mn = bev_q.U_S ./ 1e6

    bev_q_view = select(bev_q, :date_label, :window,
                        :V_U_mn, :V_S_mn, :U_U_mn, :U_S_mn,
                        :ur_U, :ur_S, :theta_U, :theta_S)
    show(stdout, MIME("text/plain"), bev_q_view; allrows = true, allcols = true)
    println()
    display(bev_q_view)
end


In [ ]:
# -----------------------------------------------------------------------------
# CELL D2.  Re-measure skilled_share and training_share using DIFFERENT denominators
# -----------------------------------------------------------------------------
# Why: the moment spec puts both in the denominator of the *labour force*.
# But the model thinks of these as population shares.  When LF participation falls
# (especially in COVID), the LF denominator gets re-composed mechanically.
# We rebuild the same stocks with FOUR different denominators and see which moves.

if cps_basic !== nothing
    rows = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), cps_basic)
        isempty(sub) && continue
        ww  = Float64.(coalesce.(sub.WTFINL, 0.0))

        in_lf      = coalesce.(sub.in_lf, false)
        is_skilled = sub.skilled
        in_train   = hasproperty(sub, :in_training) ? coalesce.(sub.in_training, false) : falses(nrow(sub))

        # Denominator 1: LF (current spec)
        denom_lf  = sum(ww[in_lf])
        # Denominator 2: working-age population (in CPS that's all 16-64 already)
        denom_pop = sum(ww)
        # Denominator 3: LF restricted to age 25-54 (prime-age, no education-driven NILF)
        prime_age = (sub.AGE .>= 25) .& (sub.AGE .<= 54)
        denom_lf_pa  = sum(ww[in_lf .& prime_age])
        denom_pop_pa = sum(ww[prime_age])

        # Skilled share
        ss_lf      = sum(ww[in_lf .&  is_skilled])               / denom_lf
        ss_pop     = sum(ww[is_skilled])                          / denom_pop
        ss_lf_pa   = sum(ww[in_lf .& is_skilled .& prime_age])    / denom_lf_pa
        ss_pop_pa  = sum(ww[is_skilled .& prime_age])             / denom_pop_pa

        # Training share — but using the LF restriction (current spec) AND population
        # (model-consistent), AND restricted to unemployed (closer to model story).
        in_unemp = coalesce.(sub.unemployed, false)
        ts_lf      = sum(ww[in_lf  .& in_train])               / denom_lf
        ts_pop     = sum(ww[in_train])                          / denom_pop
        ts_unemp   = denom_u = sum(ww[in_unemp]);
        ts_unemp   = denom_u > 0 ? sum(ww[in_unemp .& in_train]) / denom_u : NaN
        ts_lf_pa   = sum(ww[in_lf .& in_train .& prime_age])    / denom_lf_pa

        # LF participation overall and by skill
        lfpr_all = sum(ww[in_lf])               / sum(ww)
        lfpr_U   = sum(ww[in_lf .& .!is_skilled]) / sum(ww[.!is_skilled])
        lfpr_S   = sum(ww[in_lf .&  is_skilled])  / sum(ww[ is_skilled])

        push!(rows, (
            window     = w,
            ss_lf      = ss_lf,
            ss_pop     = ss_pop,
            ss_lf_pa   = ss_lf_pa,
            ss_pop_pa  = ss_pop_pa,
            ts_lf      = ts_lf,
            ts_pop     = ts_pop,
            ts_unemp   = ts_unemp,
            ts_lf_pa   = ts_lf_pa,
            lfpr_all   = lfpr_all,
            lfpr_U     = lfpr_U,
            lfpr_S     = lfpr_S,
        ))
    end
    tbl = DataFrame(rows)

    # Compute Δ% baseline → crisis for each definition
    println("\nSKILLED-SHARE under alternative denominators")
    println(repeat("-", 70))
    display(select(tbl, :window, :ss_lf, :ss_pop, :ss_lf_pa, :ss_pop_pa,
                        :lfpr_all, :lfpr_U, :lfpr_S))

    println("\nTRAINING-SHARE under alternative denominators")
    println(repeat("-", 70))
    display(select(tbl, :window, :ts_lf, :ts_pop, :ts_unemp, :ts_lf_pa))

    function pctchange(d, base_w, crisis_w, col)
        b = d[d.window .== base_w, col][1]
        c = d[d.window .== crisis_w, col][1]
        100 * (c / b - 1)
    end
    println("\nΔ% baseline → crisis  (FC | COVID) — does the sign flip across denominators?")
    println(repeat("─", 70))
    for col in [:ss_lf, :ss_pop, :ss_lf_pa, :ss_pop_pa,
                :ts_lf, :ts_pop, :ts_unemp, :ts_lf_pa,
                :lfpr_all, :lfpr_U, :lfpr_S]
        fc    = pctchange(tbl, :base_fc,    :crisis_fc,    col)
        covid = pctchange(tbl, :base_covid, :crisis_covid, col)
        @printf("  %-12s  FC: %+7.2f%%   COVID: %+7.2f%%\n", String(col), fc, covid)
    end
end

In [ ]:
# -----------------------------------------------------------------------------
# CELL D3.  Decompose the COVID skilled-share rise — composition vs. true education
# -----------------------------------------------------------------------------
# Same-age comparison: within each age band, did the BA share rise?  Or is the
# aggregate move entirely a re-composition of who is in the LF?
# If within-age BA shares are flat between base_covid and crisis_covid but the
# aggregate moves a lot, the COVID skilled_share rise is mechanical.

if cps_basic !== nothing
    rows = NamedTuple[]
    age_bands = [(16, 24), (25, 34), (35, 44), (45, 54), (55, 64)]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), cps_basic)
        isempty(sub) && continue
        ww = Float64.(coalesce.(sub.WTFINL, 0.0))
        for (lo, hi) in age_bands
            mask = (sub.AGE .>= lo) .& (sub.AGE .<= hi) .& coalesce.(sub.in_lf, false)
            denom = sum(ww[mask])
            denom == 0 && continue
            ba_share = sum(ww[mask .& sub.skilled]) / denom
            push!(rows, (window = w, age_band = "$(lo)-$(hi)", lf_weight = denom, ba_share = ba_share))
        end
    end
    age_tbl = DataFrame(rows)
    display(age_tbl)

    # Compare LF weights across windows to see who left/entered the LF
    println("\n\nLF weight by age band (relative to base_covid = 1.0)")
    println(repeat("─", 70))
    base_w = filter(:window => ==(:base_covid), age_tbl)
    crisis_w = filter(:window => ==(:crisis_covid), age_tbl)
    for ab in unique(age_tbl.age_band)
        b = base_w[base_w.age_band .== ab, :lf_weight]
        c = crisis_w[crisis_w.age_band .== ab, :lf_weight]
        isempty(b) || isempty(c) && continue
        @printf("  %s   base_covid: 1.00   crisis_covid: %.3f   ΔBA share: %+.4f\n",
                ab, c[1] / b[1],
                crisis_w[crisis_w.age_band .== ab, :ba_share][1] -
                base_w[base_w.age_band .== ab, :ba_share][1])
    end
end

In [ ]:
# -----------------------------------------------------------------------------
# CELL D4.  Training-share by age and employment status
# -----------------------------------------------------------------------------
# Where exactly did the COVID training-share drop come from?
#   - 16-24 in LF (the bulk of enrolled-without-BA workers): did they exit LF?
#   - employed students vs. unemployed students: which group shrank?

if cps_basic !== nothing
    rows = NamedTuple[]
    age_bands = [(16, 24), (25, 34), (35, 64)]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), cps_basic)
        isempty(sub) && continue
        ww = Float64.(coalesce.(sub.WTFINL, 0.0))
        in_train = hasproperty(sub, :in_training) ? coalesce.(sub.in_training, false) : falses(nrow(sub))
        for (lo, hi) in age_bands
            in_age = (sub.AGE .>= lo) .& (sub.AGE .<= hi)
            n_pop      = sum(ww[in_age])
            n_lf       = sum(ww[in_age .& coalesce.(sub.in_lf, false)])
            n_train_lf = sum(ww[in_age .& coalesce.(sub.in_lf, false) .& in_train])
            n_train_emp = sum(ww[in_age .& coalesce.(sub.employed, false)   .& in_train])
            n_train_u   = sum(ww[in_age .& coalesce.(sub.unemployed, false) .& in_train])
            n_train_pop = sum(ww[in_age .& in_train])
            push!(rows, (
                window = w, age_band = "$(lo)-$(hi)",
                lfpr           = n_lf      / max(n_pop, 1),
                ts_lf          = n_train_lf / max(n_lf, 1),
                ts_pop         = n_train_pop / max(n_pop, 1),
                share_emp      = n_train_emp / max(n_train_lf, 1),  # what fraction of LF trainees are employed
                share_unemp    = n_train_u   / max(n_train_lf, 1),
            ))
        end
    end
    display(DataFrame(rows))
end

In [ ]:
# -----------------------------------------------------------------------------
# CELL D5.  Two-segment Beveridge picture, with hand-drawn fit
# -----------------------------------------------------------------------------
# Fit a power-law u^a*v = C through the base_fc points.  Then overlay each
# subsequent window and report the perpendicular distance from the base line.
# A shift OUTSIDE the base curve (positive deviation) is a matching-technology
# shift, not a movement along the curve.

if @isdefined(bev_q)
    function loglog_fit(df, ur_col, v_col)
        x = log.(df[!, ur_col])
        y = log.(df[!, v_col])
        ok = isfinite.(x) .& isfinite.(y)
        x = x[ok]; y = y[ok]
        n = length(x); n < 4 && return (NaN, NaN)
        β = hcat(ones(n), x) \ y
        return (β[1], β[2])  # intercept, slope
    end

    base = filter(:window => ==(:base_fc), bev_q)
    for col_pair in (("ur_U", "V_U_mn"), ("ur_S", "V_S_mn"))
        ur_col, v_col = Symbol(col_pair[1]), Symbol(col_pair[2])
        a0, b0 = loglog_fit(base, ur_col, v_col)
        println("\nBeveridge fit on base_fc — log $v_col = $(round(a0, digits=2)) + $(round(b0, digits=2)) * log $ur_col")
        for w in (:crisis_fc, :base_covid, :crisis_covid)
            sub = filter(:window => ==(w), bev_q)
            pred = a0 .+ b0 .* log.(sub[!, ur_col])
            resid_mean = mean(log.(sub[!, v_col]) .- pred)
            @printf("  %s     mean log-deviation from base_fc curve: %+.3f\n",
                    String(w), resid_mean)
        end
    end
    println("\nReading:  if `base_covid` is +0.30 above the base_fc curve in BOTH segments,")
    println("           then the curve has shifted *outward* — a matching-technology change,")
    println("           not a movement along the original Beveridge curve.")
end

In [ ]:
# -----------------------------------------------------------------------------
# CELL D6.  Are skilled and unskilled separation/JFR moves *symmetric* or *skill-biased*?
# -----------------------------------------------------------------------------
# In a pure demand shock both blocks move proportionally.  In a skill-biased
# shock they don't.

if !isempty(moments)
    function pct(c, b)
        isfinite(b) && b != 0 ? 100 * (c / b - 1) : NaN
    end
    m = moments
    bfc, cfc, bcv, ccv = m[:base_fc].value, m[:crisis_fc].value, m[:base_covid].value, m[:crisis_covid].value
    nm = m[:base_fc].moment
    pairs = (
        ("ur_U", "ur_S"),
        ("jfr_U", "jfr_S"),
        ("sep_rate_U", "sep_rate_S"),
        ("mean_wage_U", "mean_wage_S"),
        ("p25_wage_U",  "p25_wage_S"),
        ("p50_wage_U",  "p50_wage_S"),
        ("emp_var_U",   "emp_var_S"),
        ("emp_cm3_U",   "emp_cm3_S"),
        ("theta_U",     "theta_S"),
    )
    println("\nFC: Δ%(U) vs Δ%(S) — skill-bias check (close to 1.0 ⇒ symmetric)")
    println(repeat("─", 70))
    for (u, s) in pairs
        iU = findfirst(==(u), nm); iS = findfirst(==(s), nm)
        dU = pct(cfc[iU], bfc[iU]); dS = pct(cfc[iS], bfc[iS])
        @printf("  %-12s Δ%%U: %+7.2f   %-12s Δ%%S: %+7.2f   ratio Δ%%U / Δ%%S: %+.2f\n",
                u, dU, s, dS, dS != 0 ? dU/dS : NaN)
    end
    println("\nCOVID: same check")
    println(repeat("─", 70))
    for (u, s) in pairs
        iU = findfirst(==(u), nm); iS = findfirst(==(s), nm)
        dU = pct(ccv[iU], bcv[iU]); dS = pct(ccv[iS], bcv[iS])
        @printf("  %-12s Δ%%U: %+7.2f   %-12s Δ%%S: %+7.2f   ratio Δ%%U / Δ%%S: %+.2f\n",
                u, dU, s, dS, dS != 0 ? dU/dS : NaN)
    end
end

In [ ]:

# -----------------------------------------------------------------------------
# CELL D7.  Population vs LF training-share — why ts_pop > ts_lf
# -----------------------------------------------------------------------------
# moments.jl line 459 defines the model's training_share as
#     training_share = agg_t / total_pop
# The model treats people-in-training as a separate stock, NOT part of the LF.
# In CPS, a full-time student with no part-time job has LABFORCE = 1 (NILF),
# so the current data spec (training_share / LF) filters them out of BOTH
# numerator and denominator.  Since enrolment is concentrated in NILF (full-
# time students), the LF-restricted moment systematically under-counts
# training relative to what the model means.
#
# This cell decomposes ts into the LF and NILF pieces and shows which
# version matches the model's definition.

if cps_basic !== nothing
    rows = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), cps_basic)
        isempty(sub) && continue
        ww = Float64.(coalesce.(sub.WTFINL, 0.0))

        in_lf    = coalesce.(sub.in_lf, false)
        in_train = hasproperty(sub, :in_training) ? coalesce.(sub.in_training, false) : falses(nrow(sub))

        pop      = sum(ww)
        lf       = sum(ww[in_lf])
        nilf     = sum(ww[.!in_lf])

        n_train_total = sum(ww[in_train])
        n_train_lf    = sum(ww[in_lf  .& in_train])
        n_train_nilf  = sum(ww[.!in_lf .& in_train])

        push!(rows, (
            window           = w,
            ts_pop_MODEL     = n_train_total / pop,                  # ← model spec
            ts_lf_OLD        = n_train_lf    / lf,                   # ← old data spec
            frac_train_NILF  = n_train_nilf  / max(n_train_total, 1),  # what fraction of trainees are NILF
            NILF_rate        = nilf / pop,
        ))
    end
    df_d7 = DataFrame(rows)
    display(df_d7)

    println("\nKey reading:")
    println("  - frac_train_NILF tells you what share of all 'in training' workers are NILF.")
    println("    That mass is dropped by the LF-restricted data spec but kept by the model spec.")
    println("  - Use ts_pop_MODEL going forward.  Update moments_*.csv to match.")

    function pct(c, b)
        isfinite(b) && b != 0 ? 100 * (c / b - 1) : NaN
    end
    println("\nΔ% baseline → crisis:")
    @printf("  ts_pop_MODEL   FC: %+7.2f%%   COVID: %+7.2f%%\n",
        pct(df_d7[df_d7.window .== :crisis_fc,    :ts_pop_MODEL][1],
            df_d7[df_d7.window .== :base_fc,      :ts_pop_MODEL][1]),
        pct(df_d7[df_d7.window .== :crisis_covid, :ts_pop_MODEL][1],
            df_d7[df_d7.window .== :base_covid,   :ts_pop_MODEL][1]))
    @printf("  ts_lf_OLD      FC: %+7.2f%%   COVID: %+7.2f%%\n",
        pct(df_d7[df_d7.window .== :crisis_fc,    :ts_lf_OLD][1],
            df_d7[df_d7.window .== :base_fc,      :ts_lf_OLD][1]),
        pct(df_d7[df_d7.window .== :crisis_covid, :ts_lf_OLD][1],
            df_d7[df_d7.window .== :base_covid,   :ts_lf_OLD][1]))
end

In [ ]:

# -----------------------------------------------------------------------------
# CELL D8.  Flow moment — Pr(unemployed_t → in_training_{t+1})
# -----------------------------------------------------------------------------
# The model says training is initiated AT the moment of entering unskilled
# unemployment.  In the data, the closest analogue is the monthly transition
# rate from unemployment-without-enrolment to enrolment.  Use CPS Basic's
# panel structure (MISH > 1 with valid_match) and build the flow.
#
# A rise in this flow during COVID would say "people responded to unemployment
# by enrolling more, even if the STOCK of enrolees fell."  That is what we
# need to know to decide whether training stock can rise in the model.

if cps_basic !== nothing
    df = filter(:valid_match => identity, cps_basic)
    df = filter(:CPSIDP => p -> p > 0, df)

    # Build (key, row) lookup keyed by (CPSIDP, YEAR, MONTH).
    nextym(y, m) = m == 12 ? (y + 1, 1) : (y, m + 1)

    lookup = Dict{Tuple{Int, Int, Int}, NamedTuple}()
    for r in eachrow(df)
        in_train_r = hasproperty(r, :in_training) ? coalesce(r.in_training, false) : false
        unemp_r    = coalesce(r.unemployed, false)
        skl_r      = coalesce(r.skilled, false)
        lookup[(Int(r.CPSIDP), Int(r.YEAR), Int(r.MONTH))] = (
            in_training = in_train_r,
            unemployed  = unemp_r,
            skilled     = skl_r,
            in_lf       = coalesce(r.in_lf, false),
        )
    end

    rows = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), df)
        for sk in (false, true)   # build separately for unskilled / skilled
            ww = Float64.(coalesce.(sub.WTFINL, 0.0))
            # numerator: unemployed AND NOT in training at t, in training at t+1
            num_w = 0.0; den_w = 0.0
            for (i, r) in enumerate(eachrow(sub))
                coalesce(r.skilled, false) == sk || continue
                u_t = coalesce(r.unemployed, false)
                t_t = hasproperty(r, :in_training) ? coalesce(r.in_training, false) : false
                u_t && !t_t || continue
                y2, m2 = nextym(Int(r.YEAR), Int(r.MONTH))
                key = (Int(r.CPSIDP), y2, m2)
                next = get(lookup, key, nothing)
                next === nothing && continue
                den_w += ww[i]
                if next.in_training
                    num_w += ww[i]
                end
            end
            push!(rows, (window = w, skilled = sk,
                         flow_U_to_train = den_w > 0 ? num_w / den_w : NaN,
                         denom = den_w))
        end
    end
    df_d8 = DataFrame(rows)
    display(df_d8)

    function pct(c, b)
        isfinite(b) && b != 0 ? 100 * (c / b - 1) : NaN
    end
    for sk in (false, true)
        sub = filter(:skilled => ==(sk), df_d8)
        bfc  = sub[sub.window .== :base_fc,      :flow_U_to_train][1]
        cfc  = sub[sub.window .== :crisis_fc,    :flow_U_to_train][1]
        bcv  = sub[sub.window .== :base_covid,   :flow_U_to_train][1]
        ccv  = sub[sub.window .== :crisis_covid, :flow_U_to_train][1]
        lab  = sk ? "skilled" : "unskilled"
        @printf("\n%s  flow Pr(U_t → train_{t+1}):\n", lab)
        @printf("  base_fc:  %.4f   crisis_fc:    %.4f   Δ%%: %+7.2f%%\n",
                bfc, cfc, pct(cfc, bfc))
        @printf("  base_cv:  %.4f   crisis_covid: %.4f   Δ%%: %+7.2f%%\n",
                bcv, ccv, pct(ccv, bcv))
    end
    println("\nReading:")
    println("  Even if the STOCK of trainees fell in COVID, this FLOW can rise if")
    println("  more newly-unemployed workers chose training.  If the flow ALSO falls,")
    println("  the COVID training decline is unambiguous — c rose and τ(x) fell.")
end


In [ ]:

# -----------------------------------------------------------------------------
# CELL D9.  Model stationary identity — is crisis_covid a coherent equilibrium?
# -----------------------------------------------------------------------------
# In the model's stationary equilibrium, for any type x:
#     m_S(x) = (φ/ν) t(x).
# With total population normalised to 1, aggregating gives
#     agg_mS = (φ/ν) × agg_t
#     model_lf ≡ agg_uU + agg_eU + agg_mS = 1 − agg_t.
# Hence, using moments.jl's definitions
#     ss = agg_mS / model_lf
#     ts = agg_t  / 1
# the stationary identity is
#     ss = (φ/ν) × ts / (1 − ts)            (★)
# equivalently
#     ts_implied(ss) = ss / (φ/ν + ss).
#
# If observed (ss, ts) violates (★), the point is NOT a steady state of the
# model.  We compute the gap window by window.

if !isempty(moments) && phi_cal !== nothing && nu_est !== nothing
    φ = phi_cal.phi[1]
    ν = nu_est.nu[1]
    model_ratio = φ / ν
    @printf("Model implies φ/ν = %.4f\n\n", model_ratio)

    rows = NamedTuple[]
    for w in WINDOWS_ORDER
        haskey(moments, w) || continue
        m = moments[w]
        idx_ss = findfirst(==("skilled_share"),  string.(m.moment))
        idx_ts = findfirst(==("training_share"), string.(m.moment))
        ss = m.value[idx_ss]; ts = m.value[idx_ts]

        # Data ratio
        ss_over_ts = ss / ts

        # Model-implied ratio at this ts:  (φ/ν) / (1 − ts)
        pred_ratio = model_ratio / (1 - ts)

        # % deviation of the data ratio from the model's stationary ratio.
        gap_pct = 100 * (ss_over_ts / pred_ratio - 1)

        # ts the model would need to see, GIVEN the observed ss, to satisfy (★):
        #     ts_implied = ss / (φ/ν + ss)
        ts_implied = ss / (model_ratio + ss)

        push!(rows, (
            window           = w,
            ss               = ss,
            ts_data          = ts,
            ss_over_ts_data  = ss_over_ts,
            ss_over_ts_pred  = pred_ratio,
            gap_pct          = gap_pct,
            ts_implied_by_ss = ts_implied,
        ))
    end
    display(DataFrame(rows))
    println("\nReading:")
    println("  gap_pct = how far the observed skilled_share / training_share is from the")
    println("  stationary identity (φ/ν) × (1 + ts/(1-ts)).  A small gap = the moment pair")
    println("  is internally consistent.  A large positive gap = skilled_share is too big")
    println("  relative to training_share for any stationary equilibrium of this model.")
    println()
    println("  ts_implied_by_ss = the training_share you would need to observe to make")
    println("  the model's stationary identity hold given the OBSERVED skilled_share.")
    println("  Compare this to the actual ts_data.  The gap reveals whether the COVID")
    println("  skilled_share rise is consistent with ANY admissible training stock.")
end

In [ ]:

# -----------------------------------------------------------------------------
# CELL D10.  Decompose the COVID jfr_U rise — matching tech vs sector reallocation vs composition
# -----------------------------------------------------------------------------
# jfr_U rose +5.7% in COVID.  No demand shock can do this.  Three hypotheses:
#   (H1) matching technology shifted (remote interviewing, online boards) — μ_U rose
#   (H2) sector reallocation — unskilled jobs grew in low-touch sectors
#   (H3) selection — marginal U searchers left LF, leaving higher-quality job seekers
#
# Decompose by AGE BAND and by INDUSTRY-of-the-found-job to distinguish.

if trans_m !== nothing && cps_basic !== nothing
    # Note: trans_m is monthly aggregates by skilled × window, so it does
    # not carry age or industry detail.  We rebuild the flow at the
    # individual level from cps_basic itself.
    df = filter(:valid_match => identity, cps_basic)
    df = filter(:CPSIDP => p -> p > 0, df)

    nextym(y, m) = m == 12 ? (y + 1, 1) : (y, m + 1)
    lookup = Dict{Tuple{Int, Int, Int}, NamedTuple}()
    for r in eachrow(df)
        lookup[(Int(r.CPSIDP), Int(r.YEAR), Int(r.MONTH))] = (
            employed    = coalesce(r.employed, false),
            unemployed  = coalesce(r.unemployed, false),
            ind_jolts   = hasproperty(r, :IND_JOLTS) ? r.IND_JOLTS : missing,
        )
    end

    # ── 10a. JFR_U by age band ───────────────────────────────────────────
    age_bands = [(16, 24), (25, 34), (35, 44), (45, 54), (55, 64)]
    rows_age = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), df)
        for (lo, hi) in age_bands
            in_age = (sub.AGE .>= lo) .& (sub.AGE .<= hi)
            sk     = .!coalesce.(sub.skilled, true)   # unskilled only
            u_t    = coalesce.(sub.unemployed, false)
            mask_t = in_age .& sk .& u_t
            ww = Float64.(coalesce.(sub.WTFINL, 0.0))
            num_w = 0.0; den_w = 0.0
            for i in findall(mask_t)
                r = sub[i, :]
                y2, m2 = nextym(Int(r.YEAR), Int(r.MONTH))
                next = get(lookup, (Int(r.CPSIDP), y2, m2), nothing)
                next === nothing && continue
                den_w += ww[i]
                if next.employed
                    num_w += ww[i]
                end
            end
            push!(rows_age, (window = w, age_band = "$(lo)-$(hi)",
                             jfr_U = den_w > 0 ? num_w / den_w : NaN,
                             denom = den_w))
        end
    end
    df_d10a = DataFrame(rows_age)
    println("\njfr_U by age band:")
    display(df_d10a)

    function pct(c, b)
        isfinite(b) && b != 0 ? 100 * (c / b - 1) : NaN
    end
    println("\nΔ% jfr_U baseline → crisis by age band:")
    for ab in unique(df_d10a.age_band)
        b_fc  = df_d10a[(df_d10a.window .== :base_fc)      .& (df_d10a.age_band .== ab), :jfr_U]
        c_fc  = df_d10a[(df_d10a.window .== :crisis_fc)    .& (df_d10a.age_band .== ab), :jfr_U]
        b_cv  = df_d10a[(df_d10a.window .== :base_covid)   .& (df_d10a.age_band .== ab), :jfr_U]
        c_cv  = df_d10a[(df_d10a.window .== :crisis_covid) .& (df_d10a.age_band .== ab), :jfr_U]
        bfc = isempty(b_fc) ? NaN : b_fc[1]; cfc = isempty(c_fc) ? NaN : c_fc[1]
        bcv = isempty(b_cv) ? NaN : b_cv[1]; ccv = isempty(c_cv) ? NaN : c_cv[1]
        @printf("  %-7s FC: %+7.2f%%   COVID: %+7.2f%%\n", ab, pct(cfc, bfc), pct(ccv, bcv))
    end

    # ── 10b. JFR_U by destination industry (only successful finders) ──────
    rows_ind = NamedTuple[]
    for w in WINDOWS_ORDER
        sub = filter(:window => ==(w), df)
        sk  = .!coalesce.(sub.skilled, true)
        u_t = coalesce.(sub.unemployed, false)
        mask_t = sk .& u_t
        ww = Float64.(coalesce.(sub.WTFINL, 0.0))
        # destination industry counts (weighted)
        dest_ind = Dict{Any, Float64}()
        total_finds = 0.0
        for i in findall(mask_t)
            r = sub[i, :]
            y2, m2 = nextym(Int(r.YEAR), Int(r.MONTH))
            next = get(lookup, (Int(r.CPSIDP), y2, m2), nothing)
            next === nothing && continue
            next.employed || continue
            ind = next.ind_jolts
            dest_ind[ind] = get(dest_ind, ind, 0.0) + ww[i]
            total_finds += ww[i]
        end
        for (ind, m) in dest_ind
            push!(rows_ind, (window = w, ind_jolts = ind,
                             share_of_finds = m / max(total_finds, 1)))
        end
    end
    df_d10b = DataFrame(rows_ind)
    println("\njfr_U destination-industry mix (share of all U→E moves found):")
    pivot_inds = unstack(df_d10b, :ind_jolts, :window, :share_of_finds)
    display(pivot_inds)

    println("\nReading:")
    println("  H1 (matching tech): jfr_U rise should be broad across all age bands and industries.")
    println("  H2 (sectoral reallocation): rise concentrated in particular industries (warehouse/")
    println("       delivery/transportation).  The industry mix shifts toward those sectors.")
    println("  H3 (selection): rise concentrated in older age bands (younger / marginal seekers")
    println("       exited LF, leaving higher-attachment workers).")
end
